# Administração de Redes: Trabalho 1
## Encaminhamento estático, ARP e filtros de captura

**Mestrado em Engenharia de Redes, FCUP, 2026/27**

Este notebook reúne, por alínea, a configuração usada (**confRes**), os outputs dos terminais (**outRes**), as capturas (**capRes**) e as respostas textuais (**texRes**). As evidências são as capturas de ecrã guardadas na pasta `evidencias/`, ao lado deste ficheiro.

> **Nota sobre as capturas de pacotes.** Os ficheiros `.pcapng` não foram guardados durante a sessão. As evidências de captura são capturas de ecrã do Wireshark e do tcpdump. Se o enunciado exigir os ficheiros `.pcapng`, é preciso repetir as capturas e anexá-los.

## 0. Topologia e endereçamento

| Máquina | Interface | Endereço |
|---|---|---|
| Term1 | ens3 | 170.2.0.33/16 |
| RCis1 | f1/0 | 170.2.0.1 |
| RCis1 | f1/1 | 195.70.40.25/30 |
| RCis2 | f1/1 | 195.70.40.26/30 |
| RCis2 | f1/0 | 192.180.40.2/24 |
| RLin | ens3 | 170.2.0.3 |
| RLin | ens4 | 192.180.30.3 |
| RLin | ens5 | 192.180.40.3 |
| Term2 | ens3 | 192.180.30.44/24 |
| Term2 | ens4 | 192.180.40.44/24 |

**Rotas relevantes (configuração prévia)**

- **Term1:** `default via 170.2.0.3` (RLin) e `192.180.40.0/24 via 170.2.0.1` (RCis1).
- **Term2:** `default via 192.180.40.2` (RCis2), por ens4.
- **RCis2:** só as redes diretamente ligadas, mais a rota para `170.2.0.0/16` que cada alínea define.

O next hop de uma rota tem de estar numa rede diretamente ligada. Por isso a rota do Term1 para 192.180.40.0/24 aponta para o RCis1 (170.2.0.1), e o caminho completo é Term1, RCis1, RCis2.

Estado inicial do Term1 (`ip route`):

![Term1: tabela de rotas e primeiros traceroutes](evidencias/1b_estado_inicial_term1.png)

*Term1: tabela de rotas e primeiros traceroutes*

## 1. Conectividade

### 1a) Traceroute do Term2 para as três interfaces do RLin

**confRes / outRes** (executado no Term2):

```
traceroute -n -N 1 192.180.30.3
traceroute -n -N 1 192.180.40.3
traceroute -n -N 1 170.2.0.3
```

`-n` desativa a resolução de nomes. `-N 1` envia uma sonda de cada vez (por omissão seriam 16 em paralelo), o que mantém as capturas ordenadas.

![Traceroutes do Term2 para as três interfaces do RLin](evidencias/1a_traceroute_term2_rlin.png)

*Traceroutes do Term2 para as três interfaces do RLin*

**texRes**

- **192.180.30.3** e **192.180.40.3**: 1 salto cada. O Term2 está diretamente ligado às duas sub-redes (ens3 e ens4).
- **170.2.0.3**: 3 saltos, Term2 → RCis2 (192.180.40.2) → RCis1 (195.70.40.25) → RLin (170.2.0.3). O Term2 não tem rota específica e usa o default gateway (RCis2). O RCis2 tinha a rota estática para 170.2.0.0/16 via RCis1.
- Em cada salto aparece o IP da interface por onde o pacote **entrou** no router, porque o ICMP Time Exceeded sai com esse endereço.
- Os `*` na primeira sonda dos saltos 2 e 3 são consistentes com a resolução ARP do próximo salto na primeira passagem (cache fria).
- O caminho é mais longo do que o necessário: o RLin está no mesmo segmento 192.180.40.0/24 do Term2, mas o Term2 não o sabe.

### 1b) Traceroute do Term1 para as duas interfaces do Term2

Destinos: `192.180.30.44` e `192.180.40.44`, executados no Term1 em três condições da rota do RCis2 para `170.2.0.0/16`.

#### Condição (i): sem rota

**confRes (RCis2):**

```
enable
conf t
no ip route 170.2.0.0 255.255.0.0 195.70.40.25
end
show ip route
```

![RCis2 sem rota para 170.2.0.0/16: só redes diretamente ligadas](evidencias/1b_i_conf_rcis2.png)

*RCis2 sem rota para 170.2.0.0/16: só redes diretamente ligadas*

**outRes (Term1):**

![Term1 para 192.180.30.44, condição (i)](evidencias/1b_i_term1_30_44.png)

*Term1 para 192.180.30.44, condição (i)*

![Term1 para 192.180.40.44, condição (i)](evidencias/1b_i_term1_40_44.png)

*Term1 para 192.180.40.44, condição (i)*

(Os processos ficaram em `Stopped` por terem sido interrompidos com Ctrl+Z. Para a entrega, Ctrl+C ou `-m 5` dão um output mais limpo.)

#### Condição (ii): via RCis1

**confRes (RCis2):**

```
conf t
ip route 170.2.0.0 255.255.0.0 195.70.40.25
end
```

**outRes (Term1):**

![Term1 para as duas interfaces do Term2, condição (ii)](evidencias/1b_ii_term1.png)

*Term1 para as duas interfaces do Term2, condição (ii)*

#### Condição (iii): via RLin

**confRes (RCis2):**

```
conf t
no ip route 170.2.0.0 255.255.0.0 195.70.40.25
ip route 170.2.0.0 255.255.0.0 192.180.40.3
end
show ip route
```

![RCis2 com S 170.2.0.0/16 via 192.180.40.3](evidencias/1b_iii_conf_rcis2.png)

*RCis2 com S 170.2.0.0/16 via 192.180.40.3*

![Term1 para as duas interfaces do Term2, condição (iii)](evidencias/1b_iii_term1.png)

*Term1 para as duas interfaces do Term2, condição (iii)*

**Resumo (RCis1 ligado)**

| Condição | → 192.180.30.44 | → 192.180.40.44 |
|---|---|---|
| i. sem rota | só o salto 1 (170.2.0.3), depois `* * *` | só o salto 1 (170.2.0.1), depois `* * *` |
| ii. via RCis1 | 2 saltos, completo | 3 saltos, completo |
| iii. via RLin | 2 saltos, completo | 3 saltos, completo |

Em (iii) aparecem `*` isolados no salto 2 do primeiro traceroute, perdas pontuais. O output é, no essencial, igual ao de (ii).

### 1c) Repetição com o RCis1 desligado

O RCis1 fica desligado até ao fim do trabalho. As condições (i), (ii) e (iii) repetem-se com os mesmos comandos no RCis2.

**outRes (Term1):**

![RCis1 desligado, condição (i)](evidencias/1c_i_term1.png)

*RCis1 desligado, condição (i)*

![RCis1 desligado, condição (ii)](evidencias/1c_ii_term1.png)

*RCis1 desligado, condição (ii)*

![RCis1 desligado, condição (iii)](evidencias/1c_iii_term1.png)

*RCis1 desligado, condição (iii)*

**Resumo (RCis1 desligado)**

| Condição | → 192.180.30.44 | → 192.180.40.44 |
|---|---|---|
| i. sem rota | salto 1 (170.2.0.3), depois `* * *` | `!H` vindo de 170.2.0.33 (~3 s) |
| ii. via RCis1 | salto 1, depois `* * *` | `!H` vindo de 170.2.0.33 |
| iii. via RLin | 2 saltos, completo | `!H` vindo de 170.2.0.33 |

### 1d) Explicação de cada caso (texRes)

O traceroute só mostra o caminho de **ida**, mas só termina se houver caminho de **volta** para as respostas ICMP. Ida e volta podem ser diferentes. A volta de qualquer resposta depende das rotas do RCis2 e do default gateway do Term2 (o RCis2).

**RCis1 ligado**

| Cenário | Ida | Volta | Onde se perde |
|---|---|---|---|
| i → 30.44 | Term1 → RLin → Term2 | Term2 → RCis2 | No RCis2: sem rota para 170.2.0.0/16 nem default |
| i → 40.44 | Term1 → RCis1 → RCis2 → Term2 | RCis2/Term2 → RCis2 | Responde o RCis1 (salto 1). O RCis2 e o Term2 não conseguem responder ao Term1 |
| ii → ambos | como acima | Term2 → RCis2 → RCis1 → Term1 | Não se perde |
| iii → ambos | como acima | Term2 → RCis2 → RLin → Term1 | Não se perde |

**RCis1 desligado**

| Cenário | Ida | Volta | Onde se perde |
|---|---|---|---|
| i → 30.44 | Term1 → RLin → Term2 | Term2 → RCis2 | No RCis2, sem rota |
| ii → 30.44 | Term1 → RLin → Term2 | Term2 → RCis2 → 195.70.40.25 (desligado) | No RCis2: a rota continua instalada, mas o next hop não responde ao ARP |
| iii → 30.44 | Term1 → RLin → Term2 | Term2 → RCis2 → RLin → Term1 | Não se perde |
| i, ii, iii → 40.44 | Term1 tenta 170.2.0.1 (desligado) | n/a | No próprio Term1: sem resposta ARP do next hop, o kernel gera `!H` após ~3 s. Não depende do RCis2 |

O `!H` vem de 170.2.0.33, o próprio Term1: a rota estática específica para 192.180.40.0/24 aponta para o RCis1, e nenhuma sonda chega a sair.

### 1e) Quando compensa o encaminhamento dinâmico (texRes)

Comparando (b) com (c): as rotas estáticas não reagem a falhas. A rota via RCis1 continua instalada com o router desligado, e o Term1 não passa a usar o RLin para 192.180.40.0/24, embora este fosse alcançável. O encaminhamento dinâmico compensa quando:

- existem **caminhos redundantes** (aqui RLin e RCis1 ligam 170.2.0.0/16 ao 192.180.40.0/24);
- as **falhas são prováveis** e a recuperação tem de ser automática, sem intervenção humana;
- a **topologia muda** (equipamentos ou ligações que entram e saem);
- a rede é **grande**, e configurar e manter rotas manualmente é trabalhoso e sujeito a erros;
- é preciso **convergência rápida**.

Em redes pequenas, estáveis e sem redundância, as rotas estáticas bastam e evitam o tráfego de controlo, o consumo de CPU e memória, a complexidade de configuração e a superfície de ataque dos protocolos de routing.

### 1f) Traceroute do Term2 para o Term1 (RCis1 desligado)

**confRes / outRes** (Term2):

```
traceroute -n -N 1 170.2.0.33
```

**Condição (i), sem rota no RCis2** (inclui o `ip route` do Term2):

![Term2 para 170.2.0.33, condição (i)](evidencias/1f_i_term2.png)

*Term2 para 170.2.0.33, condição (i)*

**Condição (ii), via RCis1 (desligado):**

![Term2 para 170.2.0.33, condição (ii)](evidencias/1f_ii_term2.png)

*Term2 para 170.2.0.33, condição (ii)*

**Condição (iii), via RLin:**

![Term2 para 170.2.0.33, condição (iii)](evidencias/1f_iii_term2.png)

*Term2 para 170.2.0.33, condição (iii)*

**Leitura**

- **(i):** o salto 1 é o RCis2 e o salto 2 devolve de 192.180.40.2 um `!H` (destination unreachable). Aqui o aviso consegue chegar ao Term2 porque este está diretamente ligado ao RCis2, ao contrário do que acontecia em 1b.
- **(ii):** o salto 1 responde (RCis2), depois só `* * *`. A rota existe, mas o next hop está desligado, por isso o RCis2 descarta sem avisar o Term2.
- **(iii):** salto 1 (RCis2) e salto 2 (RLin, 192.180.40.3), depois `* * *`. A ida chega ao Term1, mas a volta falha: o Term1 responde a 192.180.40.44 pela rota específica via RCis1, que está desligado.

### 1g) Seria útil o encaminhamento dinâmico? (texRes)

Sim. Nas três condições o traceroute do Term2 para o Term1 só seria completo com caminho nos dois sentidos, e as rotas estáticas não se ajustam à falha do RCis1:

- em (ii), o RCis2 continua a encaminhar para um next hop inativo;
- em (iii), o pacote chega ao Term1, mas a resposta fica sem retorno porque a rota específica do Term1 para 192.180.40.0/24 aponta para o RCis1 desligado.

Com um protocolo dinâmico nos routers, o RCis2 detetaria que o RCis1 desapareceu, retiraria a rota e passaria a usar o caminho via RLin.

**Ressalva:** o Term1 é um terminal e não corre protocolo de routing. A parte da volta que falha no Term1 (rota específica via RCis1) resolve-se por configuração, por exemplo usando apenas o default gateway RLin, que conhece 192.180.40.0/24. O dinâmico resolve o lado dos routers.

### 1h) Configuração final do RCis2

Rota via RLin, que fica ativa até ao fim do trabalho (a rota via RCis1 foi removida):

```
conf t
no ip route 170.2.0.0 255.255.0.0 195.70.40.25
ip route 170.2.0.0 255.255.0.0 192.180.40.3
end
show ip route
```

Ver a evidência `1b_iii_conf_rcis2.png` na alínea 1b.

### 1i) Captura e traceroute para um IP inexistente

**confRes / outRes** (Term2), com captura na ligação do Term2 à sub-rede 192.180.40.0/24 (ens4):

```
ip route get 170.2.0.200
traceroute -n -N 1 170.2.0.200
ip route get 170.2.0.200
traceroute -n -N 1 170.2.0.200
```

![Dois traceroutes seguidos para 170.2.0.200 e os dois ip route get](evidencias/1i_term2_traceroutes.png)

*Dois traceroutes seguidos para 170.2.0.200 e os dois ip route get*

**capRes:**

![Captura na ens4 do Term2, com o ICMP Redirect no pacote 9](evidencias/1i_captura_redirect.png)

*Captura na ens4 do Term2, com o ICMP Redirect no pacote 9*

**texRes (i): diferença entre os dois outputs**

| | 1.º traceroute | 2.º traceroute |
|---|---|---|
| Salto 1 | 192.180.40.2 (RCis2) | 192.180.40.3 (RLin) |
| Salto 2 | 192.180.40.3 (RLin), Time Exceeded | 192.180.40.3 (RLin), `!H` |
| Salto 3 | 192.180.40.3 (RLin), `!H` | n/a |

O segundo traceroute tem um salto a menos: o RCis2 deixou de aparecer.

**texRes (ii): explicação**

1. Antes do 1.º traceroute, o Term2 usa o default gateway: `170.2.0.200 via 192.180.40.2`.
2. As sondas de TTL 1 morrem no RCis2 (Time Exceeded). A sonda de TTL 2 é encaminhada pelo RCis2 para o RLin (192.180.40.3), que está **na mesma sub-rede e na mesma interface (f1/0)** por onde o pacote entrou.
3. Nesta situação o RCis2 envia ao Term2 um **ICMP Redirect** (pacote 9, "Redirect for network"), a indicar que existe um caminho melhor diretamente pelo RLin.
4. O Term2 aceita o redirect e guarda uma entrada em cache, visível no 2.º `ip route get`: `via 192.180.40.3 ... cache <redirected> expires 273sec`.
5. No 2.º traceroute os pacotes vão diretamente ao RLin, e o RCis2 sai do caminho. A entrada é só para este destino e expira ao fim de alguns minutos. Com outro IP da rede 170.2.0.0/16 o fenómeno repete-se.

Os `!H` de ~3 s ocorrem porque o IP 170.2.0.200 não existe: o RLin tenta fazer ARP na ens3, ninguém responde, e devolve *host unreachable* ao fim de ~3 s. Os `*` no salto 2 do 1.º traceroute são compatíveis com o limite de taxa de mensagens ICMP do Linux (uma por segundo por destino).

## 2. ARP

Captura na ligação do Term2 ao switch da 192.180.40.0/24. Antes de cada ping, a cache ARP do Term2 deve ser limpa com `ip neigh flush all`.

### 2a) Ping para um IP sem máquina

**confRes:** `ping -c 1 192.180.40.55`

**capRes:**

![Três ARP Requests em broadcast para 192.180.40.55, sem resposta](evidencias/2a_captura_arp_inexistente.png)

*Três ARP Requests em broadcast para 192.180.40.55, sem resposta*

### 2b) Ping para um IP existente

**confRes:** `ping -c 1 192.180.40.3`

**capRes:**

![Echo Request/Reply e trocas ARP posteriores](evidencias/2b_captura_arp_existente.png)

*Echo Request/Reply e trocas ARP posteriores*

O recorte capturado mostra os pacotes 6 a 12. Os pacotes 1 a 5, onde estaria a resolução ARP inicial, não estão visíveis. Se o enunciado exigir a sequência completa, convém repetir a captura com `ip neigh flush all` antes do ping.

In [1]:
# Intervalos entre ARP Requests da captura 2a (timestamps em segundos, do Wireshark)
t = [0.000000, 1.004022, 2.027919]
intervalos = [round(b - a, 3) for a, b in zip(t, t[1:])]
print('Intervalos entre ARP Requests (s):', intervalos)
print('Tempo total até desistir (s):', round(t[-1] - t[0], 3))
print('Valores por omissão no Linux: mcast_solicit = 3, retrans_time_ms = 1000')

Intervalos entre ARP Requests (s): [1.004, 1.024]
Tempo total até desistir (s): 2.028
Valores por omissão no Linux: mcast_solicit = 3, retrans_time_ms = 1000


### 2c) Análise (texRes)

O ARP resolve o endereço IP de uma máquina da mesma sub-rede no respetivo endereço MAC, necessário para construir a trama Ethernet.

- **Pedido:** o emissor envia um ARP Request em **broadcast** (MAC de destino `ff:ff:ff:ff:ff:ff`), a perguntar quem tem o IP e a quem responder.
- **Resposta:** só a máquina com esse IP responde, com um ARP Reply em **unicast** para o MAC do emissor, que traz o seu MAC.
- **Cache:** o resultado fica guardado na cache ARP durante algum tempo. Pings seguintes não geram novos pedidos, e as entradas são revalidadas mais tarde por novas trocas (unicast, estados STALE/DELAY/PROBE), como se vê nos pacotes 8 a 11 da captura 2b.
- **IP inexistente (192.180.40.55):** sem resposta, o Linux repete o pedido **3 vezes** com intervalo de cerca de **1 s** (ver cálculo acima, `mcast_solicit` e `retrans_time_ms`) e desiste. O ICMP Echo Request **nunca é enviado**, porque sem MAC de destino não há trama para o transportar. O `ping` termina com *Destination Host Unreachable*.
- **IP existente (192.180.40.3):** o ARP completa-se em poucos milissegundos e o ICMP Echo Request/Reply decorre normalmente (Echo Reply cerca de 0,6 ms depois do Request).

## 3. Filtros de captura (ssh do Term1 para 192.180.30.44)

No Term2, o sshd ativo (`systemctl start sshd.service`, se necessário). No Term1, `ssh root@192.180.30.44`, e dentro da sessão o loop do enunciado:

```
a='#'; while true; do echo $a; sleep 1; a=$a'#'; done
```

### 3a) tcpdump no Term2, todas as interfaces

**confRes:**

```
tcpdump -l -n -i any 'tcp and tcp[tcpflags] & tcp-push != 0 and ip[2:2] < 128'
```

- `tcp`: só segmentos TCP.
- `tcp[tcpflags] & tcp-push != 0`: só segmentos com a flag PUSH ativa.
- `ip[2:2] < 128`: lê 2 bytes a partir do byte 2 do cabeçalho IP (Total Length) e exige comprimento total inferior a 128.

**capRes:**

![tcpdump no Term2: ida pela ens3 (In) e volta pela ens4 (Out)](evidencias/3a_tcpdump.png)

*tcpdump no Term2: ida pela ens3 (In) e volta pela ens4 (Out)*

**Leitura:** a ida (Term1 → Term2) entra pela **ens3**, via RLin. A volta sai pela **ens4**, em direção ao RCis2 (default gateway), que a reencaminha para o RLin. É encaminhamento **assimétrico**: o Term2 só tem o default gateway por ens4. Esta volta só funciona porque o RCis2 tem a rota para 170.2.0.0/16 via RLin (alínea 1h).

### 3b) Wireshark, filtro de visualização

Captura na ligação do Term1 à rede 170.2.0.0/16.

**confRes (display filter):**

```
tcp && tcp.flags.push == 1 && ip.len < 128
```

**capRes:**

![Wireshark: dois pacotes SSH com PUSH e ip.len inferior a 128](evidencias/3b_wireshark.png)

*Wireshark: dois pacotes SSH com PUSH e ip.len inferior a 128*

O filtro de captura (BPF) decide o que é guardado. O filtro de visualização apenas esconde pacotes já capturados.

In [2]:
# 3c) Tamanho do campo de dados no pacote IP, com os valores reais da captura
frame_len = 102          # coluna Length do Wireshark (trama Ethernet)
eth_hdr   = 14
ip_len    = frame_len - eth_hdr          # ip.len (Total Length)
ihl       = 5                            # palavras de 32 bits
ip_hdr    = ihl * 4                      # ip.hdr_len
tcp_off   = 8                            # Data Offset (com opcao de timestamp)
tcp_hdr   = tcp_off * 4                  # tcp.hdr_len

dados_ip  = ip_len - ip_hdr              # payload do pacote IP (segmento TCP)
dados_tcp = dados_ip - tcp_hdr           # dados da aplicacao (tcp.len)

print(f'ip.len        = {ip_len} bytes')
print(f'ip.hdr_len    = {ip_hdr} bytes (IHL={ihl})')
print(f'dados do IP   = {dados_ip} bytes')
print(f'tcp.hdr_len   = {tcp_hdr} bytes (Data Offset={tcp_off})')
print(f'dados TCP     = {dados_tcp} bytes (confere com len=36 do Wireshark/tcpdump)')
print(f'Limite do filtro: ip.len < 128  ->  dados TCP < {128 - ip_hdr - tcp_hdr} bytes')

ip.len        = 88 bytes
ip.hdr_len    = 20 bytes (IHL=5)
dados do IP   = 68 bytes
tcp.hdr_len   = 32 bytes (Data Offset=8)
dados TCP     = 36 bytes (confere com len=36 do Wireshark/tcpdump)
Limite do filtro: ip.len < 128  ->  dados TCP < 76 bytes


### 3c) Tamanho do campo de dados no pacote IP (texRes)

O campo de dados (payload) do pacote IP obtém-se a partir de dois campos do cabeçalho:

**dados IP = Total Length − (IHL × 4)**

O Total Length é o comprimento do pacote inteiro em bytes e o IHL é o comprimento do cabeçalho em palavras de 32 bits. Para os dados da aplicação, subtrai-se ainda o cabeçalho TCP: **dados TCP = dados IP − (Data Offset × 4)**.

No pacote capturado: `ip.len` = 88, cabeçalho IP = 20 (IHL = 5), logo **dados IP = 68 bytes**. O cabeçalho TCP tem 32 bytes (Data Offset = 8), logo **dados TCP = 36 bytes**, valor que coincide com `length 36` do tcpdump e `len=36` do Wireshark.

Com estes cabeçalhos (20 + 32), o limite `ip.len < 128` corresponde a menos de 76 bytes de dados TCP.

**Observação:** só apareceram dois pacotes em cada captura. Com o loop ativo durante mais tempo, o esperado é ver pacotes de ida até as linhas passarem o limite de 128 bytes e depois desaparecerem. Uma captura mais longa ilustraria melhor o filtro.

## Dúvidas para o professor

> **Dúvida 1 (alínea 3c):** o enunciado pede o "tamanho do campo de dados no pacote IP". Devemos responder com o **payload do pacote IP** (Total Length − cabeçalho IP = 68 bytes, que inclui o cabeçalho TCP) ou com os **dados da aplicação** (36 bytes, sem cabeçalhos IP e TCP)? Neste relatório apresentámos os dois valores, mas gostaríamos de confirmar qual é o esperado.

*(Dúvida registada sobre a última pergunta trabalhada, a alínea 3. Se se referia a outra, basta ajustar esta secção.)*